In [ ]:
CREATE FILE FORMAT IF NOT EXISTS csvformat2 
    SKIP_HEADER = 1 
    TYPE = 'CSV';

-- create external stage with the csv format to stage the diamonds dataset
CREATE STAGE IF NOT EXISTS diamond_assets 
    FILE_FORMAT = csvformat2 
    URL = 's3://sfquickstarts/intro-to-machine-learning-with-snowpark-ml-for-python/diamonds.csv';

CREATE OR REPLACE TABLE DIAMONDS (
	CARAT NUMBER(38,2),
	CUT VARCHAR(16777216),
	COLOR VARCHAR(16777216),
	CLARITY VARCHAR(16777216),
	DEPTH NUMBER(38,1),
	"TABLE" NUMBER(38,1),
	PRICE NUMBER(38,0),
	X NUMBER(38,2),
	Y NUMBER(38,2),
	Z NUMBER(38,2)
);

COPY INTO DIAMONDS
FROM @DIAMOND_ASSETS;



# Welcome to the Notebooks Container Runtime!

In this notebook, we will go through the basics of using Notebooks Container Runtime. We will install packages, load data, train a model, and look at logs. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Add a query tag to the session. This helps with debugging and performance monitoring.
session.query_tag = {"origin":"sf_sit-is", "name":"aiml_notebooks_container_runtime", "version":{"major":1, "minor":0}, "attributes":{"is_quickstart":1, "source":"notebook"}}


The Container Runtime for Snowflake Notebooks includes pre-installed common packages including SnowparkML and other OSS packages.

In [ ]:
!pip freeze

Notebooks Container Runtime, along with External Access Integrations give us the flexibility to `pip install` packages from anywhere, including popular package repositories such as pypi. You can install whatever packages you need by running `!pip install <package_name>` directly in the Notebook.

We have configured this notebook to allow pypi urls with an External Access Integration. 

In [ ]:
!pip install seaborn

Just like Notebooks on the Warehouse Runtime, we can intermingle both SQL and Python cells:

In [ ]:
show tables;

Let's visualize some of our data using the `seaborn` package that we installed above:

In [ ]:
diamonds_df = session.table("DIAMONDS")
diamonds_df.show()

In [ ]:
from snowflake.ml.data.data_connector import DataConnector
data_connector = DataConnector.from_dataframe(diamonds_df)
df = data_connector.to_pandas()

import seaborn as sns

# Create a visualization
sns.histplot(
    data=df,
    x="PRICE"
)

Now, let's train a basic `XGBRegressor` machine learning model. The ML Container Runtime for Snowflake Notebooks includes pre-installed common packages for doing machine learning tasks, including SnowparkML and other OSS packages.

In [ ]:
import time
from xgboost import XGBRegressor

CATEGORICAL_COLUMNS = ["CUT", "COLOR", "CLARITY"]
NUMERICAL_COLUMNS = ["CARAT", "DEPTH", "X", "Y", "Z"]
LABEL_COLUMNS = ['PRICE']

model = XGBRegressor(max_depth=400)

t0 = time.time()
model.fit(df[NUMERICAL_COLUMNS], df[LABEL_COLUMNS])
t1 = time.time()

print(f"Fit in {t1-t0} seconds.")

In [ ]:
# Import necessary libraries
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np
import streamlit as st
import altair as alt
import pandas as pd

# Create train/test split using the existing data
from sklearn.model_selection import train_test_split
X = df[NUMERICAL_COLUMNS]
y = df[LABEL_COLUMNS]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model on training data
model = XGBRegressor(max_depth=400)
model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

# Display metrics using Streamlit
st.header("Model Performance Metrics")

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Train RMSE", f"${train_rmse:.2f}")
with col2:
    st.metric("Test RMSE", f"${test_rmse:.2f}")
with col3:
    st.metric("Train R²", f"{train_r2:.3f}")
with col4:
    st.metric("Test R²", f"{test_r2:.3f}")

# Create scatter plot of predicted vs actual values
test_results = pd.DataFrame({
    'Actual Price': y_test['PRICE'],
    'Predicted Price': y_pred_test
})

scatter_chart = alt.Chart(test_results).mark_circle().encode(
    x=alt.X('Actual Price', title='Actual Price ($)'),
    y=alt.Y('Predicted Price', title='Predicted Price ($)'),
).properties(
    title='Predicted vs Actual Diamond Prices',
    width=600,
    height=400
)

# Add the perfect prediction line
line = alt.Chart(pd.DataFrame({'x': [0, test_results['Actual Price'].max()]})).mark_line(
    color='red',
    strokeDash=[5, 5]
).encode(
    x='x',
    y='x'
)

# Display the combined chart
st.altair_chart(scatter_chart + line)

# Feature importance plot
feature_importance = pd.DataFrame({
    'Feature': NUMERICAL_COLUMNS,
    'Importance': model.feature_importances_
})

importance_chart = alt.Chart(feature_importance).mark_bar().encode(
    x=alt.X('Importance', title='Feature Importance'),
    y=alt.Y('Feature', sort='-x', title='Feature Name')
).properties(
    title='Feature Importance',
    width=600,
    height=300
)

st.altair_chart(importance_chart)
